# Image Steganography - Dataset Preparation

## 1. Cover Image Preprocessing

This notebook prepares original images for steganography
experiments and future machine learning analysis.

The preprocessing steps include:

- Loading original images
- Converting images to RGB
- Saving images in lossless PNG format
- Calculating image capacity
- Generating dataset metadata


In [5]:
from pathlib import Path
from PIL import Image, ImageOps
import csv

# Define the project directory
project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

# Define input and output directories
raw_dir = project_root / "dataset" / "raw"
cover_dir = project_root / "dataset" / "cover_png"

raw_dir.mkdir(parents=True, exist_ok=True)
cover_dir.mkdir(parents=True, exist_ok=True)

supported_extensions = {
    ".jpg", ".jpeg", ".png", ".webp", ".bmp"
}

image_files = sorted(
    path for path in raw_dir.iterdir()
    if path.suffix.lower() in supported_extensions
)

records = []

for index, image_path in enumerate(image_files):

    # Load and normalize the original image
    with Image.open(image_path) as image:
        rgb_image = ImageOps.exif_transpose(image).convert("RGB")

        width, height = rgb_image.size

        output_name = f"cover_{index:04d}.png"
        output_path = cover_dir / output_name

        # Save the normalized cover image
        rgb_image.save(output_path)

    # Calculate theoretical V2 message capacity
    total_channels = width * height * 3
    capacity_bytes = max(0, (total_channels - 112) // 8)

    records.append({
        "source_id": index,
        "original_file": image_path.name,
        "cover_file": output_name,
        "width": width,
        "height": height,
        "capacity_bytes": capacity_bytes
    })

# Save metadata as a CSV file
metadata_path = project_root / "dataset" / "metadata.csv"

with open(metadata_path, "w", newline="", encoding="utf-8") as file:

    writer = csv.DictWriter(
        file,
        fieldnames=[
            "source_id",
            "original_file",
            "cover_file",
            "width",
            "height",
            "capacity_bytes"
        ]
    )

    writer.writeheader()
    writer.writerows(records)

print("Images found:", len(image_files))
print("Images prepared:", len(records))
print("Metadata saved:", metadata_path)

Images found: 21
Images prepared: 21
Metadata saved: c:\Users\hp\Projects\image-steganography\dataset\metadata.csv


In [6]:
from pathlib import Path
from PIL import Image
import csv

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

cover_dir = project_root / "dataset" / "cover_png"
metadata_path = project_root / "dataset" / "metadata.csv"

cover_images = sorted(cover_dir.glob("*.png"))

with open(metadata_path, "r", encoding="utf-8") as file:
    records = list(csv.DictReader(file))

print("Cover images:", len(cover_images))
print("Metadata records:", len(records))

for image_path in cover_images[:5]:
    with Image.open(image_path) as image:
        print(image_path.name, image.size, image.mode)

assert len(cover_images) == len(records) > 0

print("Dataset verification passed.")

Cover images: 21
Metadata records: 21
cover_0000.png (736, 708) RGB
cover_0001.png (681, 681) RGB
cover_0002.png (712, 1200) RGB
cover_0003.png (735, 856) RGB
cover_0004.png (736, 1204) RGB
Dataset verification passed.


## 2. Stego Dataset Generation

This section creates an initial labeled dataset for steganalysis.

Each source image is standardized to 256 × 256 pixels.
A corresponding stego image is generated using either
Sequential LSB or PRNG-based LSB substitution.

The V2 message format is used for compatibility with
the extraction module.

Labels:
- 0: Cover image
- 1: Stego image

The initial dataset is intended for pipeline testing,
not for final machine learning evaluation.

In [7]:
from pathlib import Path
from PIL import Image, ImageOps
import random
import string
import zlib
import csv

# Locate the project directory
project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

source_dir = project_root / "dataset" / "cover_png"
experiment_cover_dir = project_root / "dataset" / "experiment_cover"
stego_dir = project_root / "dataset" / "stego_png"

experiment_cover_dir.mkdir(parents=True, exist_ok=True)
stego_dir.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE = (256, 256)
HEADER_BITS = 112
SEED = 12345


def convert_to_bits(data):
    """Convert bytes to bits in MSB-first order."""
    return [
        (byte >> bit_index) & 1
        for byte in data
        for bit_index in range(7, -1, -1)
    ]


def generate_stego_image(cover_path, message, output_path, method, seed):
    """Generate a V2-compatible spatial-domain LSB stego image."""

    with Image.open(cover_path) as image:
        rgb = image.convert("RGB")
        values = bytearray(rgb.tobytes())
        size = rgb.size

    payload = message.encode("utf-8")

    method_id = 0 if method == "sequential" else 1

    header = (
        b"STEG"
        + bytes([2, method_id])
        + len(payload).to_bytes(4, "big")
        + zlib.crc32(payload).to_bytes(4, "big")
    )

    header_bits = convert_to_bits(header)
    payload_bits = convert_to_bits(payload)

    if len(header_bits) + len(payload_bits) > len(values):
        raise ValueError("Message exceeds image capacity.")

    # Embed the V2 header sequentially
    for i, bit in enumerate(header_bits):
        values[i] = (values[i] & 254) | bit

    # Select payload positions
    if method == "sequential":
        positions = range(HEADER_BITS, HEADER_BITS + len(payload_bits))
    else:
        positions = random.Random(seed).sample(
            range(HEADER_BITS, len(values)),
            len(payload_bits)
        )

    # Embed payload bits
    for position, bit in zip(positions, payload_bits):
        values[position] = (values[position] & 254) | bit

    output_image = Image.frombytes("RGB", size, bytes(values))
    output_image.save(output_path)


# Read previously prepared images
source_images = sorted(source_dir.glob("*.png"))

records = []

for index, source_path in enumerate(source_images):

    # Standardize the image size
    with Image.open(source_path) as image:
        cover = ImageOps.fit(
            image.convert("RGB"),
            IMAGE_SIZE,
            method=Image.Resampling.LANCZOS
        )

    cover_name = f"cover_{index:04d}.png"
    cover_path = experiment_cover_dir / cover_name
    cover.save(cover_path)

    # Alternate between the two LSB algorithms
    method = "sequential" if index % 2 == 0 else "prng"

    # Use different message lengths
    payload_length = [512, 2048, 4096][index % 3]

    rng = random.Random(1000 + index)
    alphabet = string.ascii_letters + string.digits

    message = "".join(
        rng.choices(alphabet, k=payload_length)
    )

    stego_name = f"stego_{index:04d}_{method}.png"
    stego_path = stego_dir / stego_name
    image_seed = SEED + index

    generate_stego_image(
        cover_path,
        message,
        stego_path,
        method,
        image_seed
    )

    # Record the cover image
    records.append({
        "source_id": index,
        "file": str(cover_path.relative_to(project_root)),
        "label": 0,
        "method": "none",
        "seed": "",
        "message": ""
    })

    # Record the corresponding stego image
    records.append({
        "source_id": index,
        "file": str(stego_path.relative_to(project_root)),
        "label": 1,
        "method": method,
        "seed": image_seed,
        "message": message
    })


# Save ground-truth metadata
manifest_path = project_root / "dataset" / "stego_manifest.csv"

with open(manifest_path, "w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(
        file,
        fieldnames=[
            "source_id", "file", "label",
            "method", "seed", "message"
        ]
    )
    writer.writeheader()
    writer.writerows(records)

print("Source images:", len(source_images))
print("Cover samples:", sum(r["label"] == 0 for r in records))
print("Stego samples:", sum(r["label"] == 1 for r in records))
print("Total samples:", len(records))
print("Manifest saved:", manifest_path)


Source images: 21
Cover samples: 21
Stego samples: 21
Total samples: 42
Manifest saved: c:\Users\hp\Projects\image-steganography\dataset\stego_manifest.csv


In [8]:
from pathlib import Path
from collections import Counter

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

ucid_dir = project_root / "dataset" / "ucid_raw" / "UCID-1338"

image_files = [
    p for p in ucid_dir.rglob("*")
    if p.is_file()
]

print("Total files:", len(image_files))
print("File formats:", Counter(p.suffix.lower() for p in image_files))
print("Sample files:", [p.name for p in image_files[:5]])

Total files: 1338
File formats: Counter({'.tif': 1338})
Sample files: ['1.tif', '10.tif', '100.tif', '1000.tif', '1001.tif']


## 3. UCID Dataset Preparation

This section prepares a reproducible subset of the UCID dataset
for machine learning experiments.

The workflow includes:

- Selecting 300 source images
- Converting images into RGB PNG format
- Standardizing image dimensions to 256 x 256
- Assigning images to training, validation, and test sets
- Saving dataset metadata for future experiments

In [9]:
from pathlib import Path
from PIL import Image, ImageOps
import random
import csv

# Locate the project root
project_root = Path.cwd().resolve()

if project_root.name == "notebooks":
    project_root = project_root.parent

# Define input and output directories
ucid_dir = project_root / "dataset" / "ucid_raw" / "UCID-1338"
output_dir = project_root / "dataset" / "ucid_cover"

output_dir.mkdir(parents=True, exist_ok=True)

# Find all TIFF images
image_files = sorted(
    path for path in ucid_dir.rglob("*")
    if path.is_file()
    and path.suffix.lower() in {".tif", ".tiff"}
)

assert len(image_files) >= 300, "Not enough UCID images."

# Select 300 reproducible samples
rng = random.Random(42)
rng.shuffle(image_files)

selected_images = image_files[:300]

# Define dataset split sizes
train_count = 210
validation_count = 45

records = []

for index, image_path in enumerate(selected_images):

    # Assign each original image to one split
    if index < train_count:
        split = "train"

    elif index < train_count + validation_count:
        split = "validation"

    else:
        split = "test"

    # Convert the original image to RGB PNG
    with Image.open(image_path) as image:

        rgb = ImageOps.fit(
            image.convert("RGB"),
            (256, 256),
            method=Image.Resampling.LANCZOS
        )

        output_name = f"ucid_cover_{index:04d}.png"
        output_path = output_dir / output_name

        rgb.save(output_path)

    records.append({
        "source_id": f"ucid_{index:04d}",
        "original_file": image_path.name,
        "cover_file": str(
            output_path.relative_to(project_root)
        ),
        "split": split,
        "width": 256,
        "height": 256
    })

# Save dataset metadata
metadata_path = project_root / "dataset" / "ucid_metadata.csv"

with open(metadata_path, "w", newline="", encoding="utf-8") as file:

    writer = csv.DictWriter(
        file,
        fieldnames=[
            "source_id",
            "original_file",
            "cover_file",
            "split",
            "width",
            "height"
        ]
    )

    writer.writeheader()
    writer.writerows(records)

print("Total UCID images:", len(image_files))
print("Prepared images:", len(records))

for split in ["train", "validation", "test"]:
    count = sum(record["split"] == split for record in records)
    print(f"{split.capitalize()}: {count}")

print("Metadata saved:", metadata_path)

Total UCID images: 1338
Prepared images: 300
Train: 210
Validation: 45
Test: 45
Metadata saved: C:\Users\hp\Projects\image-steganography\dataset\ucid_metadata.csv


## 4. Main Stego Dataset Generation

This section generates stego images from the prepared UCID covers.

Two embedding methods are used:
- Sequential 1-bit LSB
- PRNG-based 1-bit LSB

Three payload rates are evaluated: 5%, 15%, and 30%
of available RGB channel LSB capacity.

The V2 protocol is preserved for compatibility with
the extraction module.

Each source image retains its original train,
validation, or test assignment.

The generated metadata will be used for machine
learning experiments and extraction verification.

In [10]:
from pathlib import Path
from PIL import Image
import csv
import random
import string
import zlib
import hashlib
from collections import Counter

# Locate the project root
project_root = Path.cwd().resolve()

if project_root.name == "notebooks":
    project_root = project_root.parent

# Define input and output locations
metadata_path = project_root / "dataset" / "ucid_metadata.csv"
stego_dir = project_root / "dataset" / "ucid_stego"
manifest_path = project_root / "dataset" / "ucid_manifest.csv"

stego_dir.mkdir(parents=True, exist_ok=True)

# Define the experimental settings
HEADER_BITS = 112
PAYLOAD_RATES = [0.05, 0.15, 0.30]
METHODS = ["sequential", "prng"]
ALPHABET = string.ascii_letters + string.digits

# Start with 10 images, then change this to 300
MAX_SOURCES = 300


def bytes_to_binary_bits(data):
    """Convert bytes into MSB-first binary bits."""
    return [
        (byte >> bit_index) & 1
        for byte in data
        for bit_index in range(7, -1, -1)
    ]


def create_v2_stego(cover_bytes, size, message, method, seed):
    """Embed a UTF-8 message using the V2 protocol."""

    values = bytearray(cover_bytes)
    payload = message.encode("utf-8")

    method_id = 0 if method == "sequential" else 1

    # Build the 14-byte V2 header
    header = (
        b"STEG"
        + bytes([2, method_id])
        + len(payload).to_bytes(4, "big")
        + zlib.crc32(payload).to_bytes(4, "big")
    )

    header_bits = bytes_to_binary_bits(header)
    payload_bits = bytes_to_binary_bits(payload)

    if len(header_bits) != HEADER_BITS:
        raise ValueError("Invalid V2 header size.")

    if HEADER_BITS + len(payload_bits) > len(values):
        raise ValueError("Payload exceeds image capacity.")

    # Store the header sequentially
    for position, bit in enumerate(header_bits):
        values[position] = (values[position] & 254) | bit

    # Select message embedding positions
    if method == "sequential":
        positions = range(
            HEADER_BITS,
            HEADER_BITS + len(payload_bits)
        )

    elif method == "prng":
        positions = random.Random(seed).sample(
            range(HEADER_BITS, len(values)),
            len(payload_bits)
        )

    else:
        raise ValueError("Unknown embedding method.")

    # Embed the message bits
    for position, bit in zip(positions, payload_bits):
        values[position] = (values[position] & 254) | bit

    return Image.frombytes("RGB", size, bytes(values))


# Read UCID source metadata
with open(metadata_path, "r", encoding="utf-8", newline="") as file:
    source_records = list(csv.DictReader(file))

selected_records = source_records[:MAX_SOURCES]

if not selected_records:
    raise ValueError("No UCID cover images found.")

manifest = []

for index, record in enumerate(selected_records):

    source_id = record["source_id"]
    split = record["split"]
    cover_path = project_root / record["cover_file"]

    # Read the prepared RGB cover image
    with Image.open(cover_path) as image:
        rgb = image.convert("RGB")
        image_size = rgb.size
        cover_bytes = rgb.tobytes()

    available_bits = len(cover_bytes) - HEADER_BITS

    # Add the original cover sample
    manifest.append({
        "source_id": source_id,
        "split": split,
        "file": record["cover_file"],
        "label": 0,
        "method": "none",
        "payload_rate": 0,
        "payload_bytes": 0,
        "seed": "",
        "message_seed": "",
        "message_sha256": ""
    })

    # Generate multiple payload sizes
    for rate_index, rate in enumerate(PAYLOAD_RATES):

        payload_bytes = int(available_bits * rate / 8)

        # Generate a reproducible secret message
        message_seed = 100000 + index * 10 + rate_index
        message_rng = random.Random(message_seed)

        message = "".join(
            message_rng.choices(
                ALPHABET,
                k=payload_bytes
            )
        )

        message_hash = hashlib.sha256(
            message.encode("utf-8")
        ).hexdigest()

        # Use the same message for both methods
        for method in METHODS:

            seed = 300000 + index * 10 + rate_index

            stego_image = create_v2_stego(
                cover_bytes,
                image_size,
                message,
                method,
                seed
            )

            filename = (
                f"{source_id}_{method}_"
                f"r{int(rate * 100):02d}.png"
            )

            output_path = stego_dir / filename
            stego_image.save(output_path)

            manifest.append({
                "source_id": source_id,
                "split": split,
                "file": str(output_path.relative_to(project_root)),
                "label": 1,
                "method": method,
                "payload_rate": rate,
                "payload_bytes": payload_bytes,
                "seed": seed if method == "prng" else "",
                "message_seed": message_seed,
                "message_sha256": message_hash
            })

# Save the labeled dataset manifest
fieldnames = [
    "source_id",
    "split",
    "file",
    "label",
    "method",
    "payload_rate",
    "payload_bytes",
    "seed",
    "message_seed",
    "message_sha256"
]

with open(manifest_path, "w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(manifest)

# Print dataset summary
cover_count = sum(row["label"] == 0 for row in manifest)
stego_count = sum(row["label"] == 1 for row in manifest)

print("Source images:", len(selected_records))
print("Cover images:", cover_count)
print("Stego images:", stego_count)
print("Total samples:", len(manifest))
print("Splits:", dict(Counter(row["split"] for row in manifest)))
print("Manifest saved:", manifest_path)

Source images: 300
Cover images: 300
Stego images: 1800
Total samples: 2100
Splits: {'train': 1470, 'validation': 315, 'test': 315}
Manifest saved: C:\Users\hp\Projects\image-steganography\dataset\ucid_manifest.csv
